# Tahap 3: Case Retrieval
**Mata Kuliah:** Penalaran Komputer — SubCPMK-3 Case-Based Reasoning  
**Domain:** Perlindungan Anak  
**Tujuan:** Membangun sistem retrieval kasus yang paling mirip dengan query baru

### Dua pendekatan yang diimplementasikan:
1. **TF-IDF + SVM** (statistik, ringan, cepat)
2. **SentenceTransformer** (embedding multilingual, lebih akurat)

### Alur Notebook:
1. Install & Import
2. Load Data dari Tahap 2
3. Preprocessing Teks
4. Pendekatan 1 — TF-IDF + SVM
5. Pendekatan 2 — SentenceTransformer Embedding
6. Fungsi `retrieve()` Terpadu
7. Pengujian Awal dengan Query Manual
8. Simpan Model & Vectors
9. Evaluasi Awal (Precision@K)

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q scikit-learn sentence-transformers pandas numpy tqdm joblib

## Cell 2 — Import Library

In [ ]:
import os
import re
import json
import logging
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
print("✅ Library berhasil diimport")

## Cell 3 — Konfigurasi

In [ ]:
CONFIG = {
    # TF-IDF
    "TFIDF_MAX_FEATURES" : 5000,
    "TFIDF_NGRAM_RANGE"  : (1, 2),
    "TFIDF_MIN_DF"       : 2,

    # SVM
    "SVM_C"              : 1.0,
    "TEST_SIZE"          : 0.2,   # 80:20 split
    "RANDOM_STATE"       : 42,

    # SentenceTransformer
    # Model multilingual ringan, support Bahasa Indonesia
    "ST_MODEL"           : "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "ST_BATCH_SIZE"      : 16,

    # Retrieval
    "TOP_K"              : 5,     # top-k kasus termirip
    "RETRIEVAL_MODE"     : "tfidf",  # "tfidf" atau "embedding"
}

# Direktori
BASE_DIR   = Path(".").resolve()
DATA_PROC  = BASE_DIR / "data" / "processed"
DATA_EVAL  = BASE_DIR / "data" / "eval"
MODEL_DIR  = BASE_DIR / "models"
LOGS_DIR   = BASE_DIR / "logs"

for d in [DATA_EVAL, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Logging
logging.basicConfig(
    level    = logging.INFO,
    format   = "%(asctime)s | %(levelname)s | %(message)s",
    handlers = [
        logging.FileHandler(LOGS_DIR / "retrieval.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
print("✅ Konfigurasi selesai")

## Cell 4 — Load Data dari Tahap 2

In [ ]:
json_path = DATA_PROC / "cases.json"
assert json_path.exists(), f"❌ {json_path} tidak ditemukan. Jalankan Tahap 2 dulu."

with open(json_path, encoding="utf-8") as f:
    cases = json.load(f)

df = pd.DataFrame(cases)
print(f"✅ {len(df)} kasus berhasil dimuat")
print(f"   Kolom: {list(df.columns)}")

# Pastikan kolom yang dibutuhkan tersedia
assert "teks_full" in df.columns,       "❌ Kolom teks_full tidak ada"
assert "amar_putusan" in df.columns,    "❌ Kolom amar_putusan tidak ada"
assert "pasal_dakwaan" in df.columns,   "❌ Kolom pasal_dakwaan tidak ada"

print(f"\n   Preview case_id: {list(df['case_id'].head(5))}")

## Cell 5 — Preprocessing Teks untuk Retrieval

In [ ]:
# Stopwords Bahasa Indonesia sederhana
STOPWORDS_ID = set([
    "yang", "dan", "di", "ke", "dari", "dengan", "untuk", "pada", "dalam",
    "adalah", "ini", "itu", "atau", "juga", "sudah", "oleh", "tidak", "ada",
    "bahwa", "telah", "akan", "maka", "serta", "para", "kepada", "karena",
    "atas", "sebagai", "dapat", "tersebut", "sehingga", "antara", "nomor",
    "halaman", "republik", "indonesia", "mahkamah", "pengadilan", "agung",
    "direktori", "putusan", "perkara", "sesuai", "majelis", "hakim",
])


def preprocess_teks(teks: str, max_chars: int = 8000) -> str:
    """
    Preprocessing ringan untuk TF-IDF:
    - lowercase
    - hapus angka & tanda baca berlebih
    - hapus stopwords
    - potong jika terlalu panjang
    """
    if not teks or not isinstance(teks, str):
        return ""

    teks = teks[:max_chars].lower()
    teks = re.sub(r"\d+", " ", teks)
    teks = re.sub(r"[^a-z\s]", " ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()

    kata = [k for k in teks.split() if k not in STOPWORDS_ID and len(k) > 2]
    return " ".join(kata)


def buat_teks_representasi(row: pd.Series) -> str:
    """
    Gabungkan beberapa field penting menjadi satu teks representasi.
    Memberikan bobot lebih pada field hukum yang kritis.
    """
    bagian = []

    # Field dengan bobot tinggi (diulang 2x)
    for field in ["pasal_dakwaan", "pasal_terbukti", "amar_putusan"]:
        val = str(row.get(field, "")).strip()
        if val:
            bagian.append(val)
            bagian.append(val)  # bobot 2x

    # Field dengan bobot normal
    for field in ["ringkasan_fakta", "barang_bukti", "terdakwa"]:
        val = str(row.get(field, "")).strip()
        if val:
            bagian.append(val)

    # Teks penuh sebagai konteks tambahan (dibatasi)
    teks_penuh = str(row.get("teks_full", ""))[:3000]
    if teks_penuh:
        bagian.append(teks_penuh)

    return " ".join(bagian)


# Buat kolom teks representasi
print("🔄 Membuat teks representasi...")
df["teks_repr"]      = df.apply(buat_teks_representasi, axis=1)
df["teks_processed"] = df["teks_repr"].apply(preprocess_teks)

# Buat label untuk SVM (berdasarkan amar putusan)
def buat_label(amar: str) -> str:
    """Buat label klasifikasi sederhana dari amar putusan."""
    amar = str(amar).lower()
    if any(k in amar for k in ["bebas", "tidak terbukti", "dibebaskan"]):
        return "bebas"
    elif any(k in amar for k in ["lepas", "tuntutan"]):
        return "lepas"
    elif any(k in amar for k in ["terbukti", "bersalah", "pidana", "penjara", "denda"]):
        return "terbukti"
    return "lainnya"

df["label"] = df["amar_putusan"].apply(buat_label)

print(f"✅ Preprocessing selesai")
print(f"\n  Distribusi label:")
print(df["label"].value_counts().to_string())
print(f"\n  Contoh teks_processed (100 char):")
print(f"  {df['teks_processed'].iloc[0][:100]}...")

## Cell 6 — Pendekatan 1: TF-IDF + SVM

> TF-IDF membangun representasi vektor dari frekuensi kata.  
> SVM dipakai untuk klasifikasi/retrieval berdasarkan label amar putusan.

In [ ]:
print("🔄 Membangun TF-IDF vectorizer...")

# Fit TF-IDF pada seluruh corpus
tfidf = TfidfVectorizer(
    max_features = CONFIG["TFIDF_MAX_FEATURES"],
    ngram_range  = CONFIG["TFIDF_NGRAM_RANGE"],
    min_df       = CONFIG["TFIDF_MIN_DF"],
    sublinear_tf = True,
)
X_tfidf = tfidf.fit_transform(df["teks_processed"])

print(f"✅ TF-IDF matrix: {X_tfidf.shape[0]} dokumen × {X_tfidf.shape[1]} fitur")

# ── Train SVM ──────────────────────────────────────────────
# Filter label yang cukup sampelnya untuk split
label_counts = df["label"].value_counts()
label_valid  = label_counts[label_counts >= 2].index.tolist()
df_svm       = df[df["label"].isin(label_valid)].copy()

print(f"\n🔄 Training SVM pada {len(df_svm)} sampel...")
print(f"   Label yang dipakai: {label_valid}")

le = LabelEncoder()
y  = le.fit_transform(df_svm["label"])

idx = df_svm.index.tolist()
X_svm = X_tfidf[idx]

if len(df_svm) >= 10:
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X_svm, y, idx,
        test_size    = CONFIG["TEST_SIZE"],
        random_state = CONFIG["RANDOM_STATE"],
        stratify     = y if len(label_valid) > 1 else None,
    )

    svm = LinearSVC(C=CONFIG["SVM_C"], max_iter=2000, random_state=CONFIG["RANDOM_STATE"])
    svm.fit(X_train, y_train)

    y_pred = svm.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)

    print(f"\n  ✅ SVM Training selesai")
    print(f"  Accuracy (test set): {acc:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred,
                                 target_names=le.classes_,
                                 zero_division=0))

    # Simpan indeks test untuk evaluasi Tahap 5
    test_indices = [df_svm.iloc[i]["case_id"] for i in range(len(idx_test))]
else:
    print(f"\n  ⚠️  Data terlalu sedikit untuk split ({len(df_svm)} sampel)")
    print(f"      SVM dilatih pada semua data tanpa test set")
    svm = LinearSVC(C=CONFIG["SVM_C"], max_iter=2000, random_state=CONFIG["RANDOM_STATE"])
    svm.fit(X_svm, y)
    acc = accuracy_score(y, svm.predict(X_svm))
    print(f"  Accuracy (train): {acc:.4f}")

logger.info(f"TF-IDF: {X_tfidf.shape}, SVM accuracy: {acc:.4f}")

## Cell 7 — Pendekatan 2: SentenceTransformer Embedding

> Model multilingual yang support Bahasa Indonesia.  
> Menghasilkan dense vector 384 dimensi per dokumen.  
> Lebih akurat dari TF-IDF untuk teks hukum yang panjang.

In [ ]:
print(f"🔄 Loading model: {CONFIG['ST_MODEL']}")
print(f"   (Download pertama kali ~90MB, harap tunggu...)\n")

st_model = SentenceTransformer(CONFIG["ST_MODEL"])

print(f"✅ Model loaded")
print(f"   Embedding dimension: {st_model.get_sentence_embedding_dimension()}")

# Encode semua dokumen
print(f"\n🔄 Encoding {len(df)} dokumen...")
embeddings = st_model.encode(
    df["teks_repr"].tolist(),
    batch_size     = CONFIG["ST_BATCH_SIZE"],
    show_progress_bar = True,
    normalize_embeddings = True,  # L2 normalize untuk cosine similarity
)

print(f"\n✅ Embedding selesai")
print(f"   Shape: {embeddings.shape}")  # (n_dokumen, 384)

logger.info(f"SentenceTransformer embeddings: {embeddings.shape}")

## Cell 8 — Fungsi `retrieve()` Terpadu

> Fungsi utama yang dipakai di Tahap 4 dan seterusnya.  
> Mode bisa dipilih: `tfidf` atau `embedding`

In [ ]:
def retrieve(query: str, k: int = None, mode: str = None) -> list:
    """
    Temukan k kasus paling mirip dengan query.

    Args:
        query : teks kasus baru yang ingin dicari padanannya
        k     : jumlah kasus yang dikembalikan (default: CONFIG['TOP_K'])
        mode  : 'tfidf' atau 'embedding' (default: CONFIG['RETRIEVAL_MODE'])

    Returns:
        List of dict berisi:
        [
          {
            'case_id'    : str,
            'rank'       : int,
            'score'      : float,
            'no_perkara' : str,
            'amar_putusan': str,
            'pasal_dakwaan': str,
            'ringkasan_fakta': str,
          },
          ...
        ]
    """
    if k    is None: k    = CONFIG["TOP_K"]
    if mode is None: mode = CONFIG["RETRIEVAL_MODE"]

    # ── Preprocessing query ─────────────────────────────────
    query_clean = preprocess_teks(query)
    if not query_clean:
        logger.warning("Query kosong setelah preprocessing")
        return []

    # ── Hitung vektor query & similarity ───────────────────
    if mode == "tfidf":
        q_vec  = tfidf.transform([query_clean])
        scores = cosine_similarity(q_vec, X_tfidf).flatten()

    elif mode == "embedding":
        q_emb  = st_model.encode([query], normalize_embeddings=True)
        scores = cosine_similarity(q_emb, embeddings).flatten()

    else:
        raise ValueError(f"Mode tidak dikenal: {mode}. Pilih 'tfidf' atau 'embedding'")

    # ── Ambil top-k indeks ──────────────────────────────────
    top_idx = np.argsort(scores)[::-1][:k]

    hasil = []
    for rank, idx in enumerate(top_idx, start=1):
        row = df.iloc[idx]
        hasil.append({
            "rank"           : rank,
            "case_id"        : row["case_id"],
            "score"          : round(float(scores[idx]), 4),
            "no_perkara"     : str(row.get("no_perkara", "")),
            "pengadilan"     : str(row.get("pengadilan", "")),
            "tanggal_putusan": str(row.get("tanggal_putusan", "")),
            "terdakwa"       : str(row.get("terdakwa", "")),
            "pasal_dakwaan"  : str(row.get("pasal_dakwaan", "")),
            "pasal_terbukti" : str(row.get("pasal_terbukti", "")),
            "amar_putusan"   : str(row.get("amar_putusan", "")),
            "vonis_penjara"  : str(row.get("vonis_penjara", "")),
            "ringkasan_fakta": str(row.get("ringkasan_fakta", "")),
            "label"          : str(row.get("label", "")),
        })

    logger.info(f"retrieve(mode={mode}, k={k}) → top score={hasil[0]['score'] if hasil else 0}")
    return hasil


def tampilkan_hasil(hasil: list, query: str = "") -> None:
    """Helper untuk print hasil retrieve secara rapi."""
    if query:
        print(f"🔍 Query: {query[:100]}...")
    print(f"{'─'*60}")
    for h in hasil:
        print(f"  #{h['rank']} [{h['case_id']}]  score={h['score']:.4f}")
        print(f"     No. Perkara : {h['no_perkara']}")
        print(f"     Terdakwa    : {h['terdakwa'][:50]}")
        print(f"     Pasal       : {h['pasal_dakwaan'][:70]}")
        print(f"     Amar        : {h['amar_putusan'][:70]}")
        print(f"     Label       : {h['label']}")
        print()


print("✅ Fungsi retrieve() siap")

## Cell 9 — Pengujian Awal dengan Query Manual

> Siapkan 5-10 query uji beserta ground-truth `case_id`.  
> Ini juga sekaligus menghasilkan `/data/eval/queries.json`.

In [ ]:
# ============================================================
# QUERY UJI MANUAL
# Isi dengan teks kasus baru (bisa dari putusan yang belum masuk corpus)
# ground_truth_ids: case_id yang seharusnya ditemukan (jika tahu)
# ============================================================
QUERIES_UJI = [
    {
        "query_id"         : "q001",
        "query_text"       : "terdakwa melakukan kekerasan seksual terhadap anak di bawah umur pasal 76D UU perlindungan anak",
        "ground_truth_ids" : [],  # isi jika tahu case_id yang relevan
        "catatan"          : "query tentang kekerasan seksual anak"
    },
    {
        "query_id"         : "q002",
        "query_text"       : "terdakwa melakukan penganiayaan terhadap anak menyebabkan luka berat pasal 80 UU 35 tahun 2014",
        "ground_truth_ids" : [],
        "catatan"          : "query tentang penganiayaan anak"
    },
    {
        "query_id"         : "q003",
        "query_text"       : "eksploitasi anak untuk tujuan komersial perdagangan anak trafficking",
        "ground_truth_ids" : [],
        "catatan"          : "query tentang eksploitasi/trafficking anak"
    },
    {
        "query_id"         : "q004",
        "query_text"       : "penelantaran anak oleh orang tua tidak memberikan nafkah dan perlindungan",
        "ground_truth_ids" : [],
        "catatan"          : "query tentang penelantaran anak"
    },
    {
        "query_id"         : "q005",
        "query_text"       : "pencabulan terhadap anak korban di bawah 12 tahun terdakwa orang dewasa",
        "ground_truth_ids" : [],
        "catatan"          : "query tentang pencabulan anak"
    },
]

# Simpan queries ke eval
queries_path = DATA_EVAL / "queries.json"
with open(queries_path, "w", encoding="utf-8") as f:
    json.dump(QUERIES_UJI, f, ensure_ascii=False, indent=2)
print(f"💾 Queries disimpan: {queries_path}")

# ── Jalankan retrieve untuk setiap query ────────────────────
print(f"\n{'='*60}")
print(f"  HASIL RETRIEVAL — MODE: {CONFIG['RETRIEVAL_MODE'].upper()}")
print(f"{'='*60}\n")

semua_hasil = []
for q in QUERIES_UJI:
    print(f"\n📌 [{q['query_id']}] {q['catatan']}")
    hasil_tfidf = retrieve(q["query_text"], k=CONFIG["TOP_K"], mode="tfidf")
    tampilkan_hasil(hasil_tfidf, q["query_text"])

    # Simpan hasil
    semua_hasil.append({
        "query_id"   : q["query_id"],
        "query_text" : q["query_text"],
        "mode"       : "tfidf",
        "top_k"      : hasil_tfidf,
    })

# Simpan hasil retrieval
retrieval_path = DATA_EVAL / "retrieval_results.json"
with open(retrieval_path, "w", encoding="utf-8") as f:
    json.dump(semua_hasil, f, ensure_ascii=False, indent=2)
print(f"\n💾 Hasil retrieval disimpan: {retrieval_path}")

## Cell 10 — Perbandingan TF-IDF vs Embedding

In [ ]:
# Bandingkan hasil top-3 dari dua mode untuk query pertama
q_test = QUERIES_UJI[0]["query_text"]

print(f"🔍 Query: {q_test}\n")
print(f"{'='*60}")
print(f"  MODE: TF-IDF")
print(f"{'='*60}")
hasil_tf = retrieve(q_test, k=3, mode="tfidf")
tampilkan_hasil(hasil_tf)

print(f"{'='*60}")
print(f"  MODE: EMBEDDING (SentenceTransformer)")
print(f"{'='*60}")
hasil_em = retrieve(q_test, k=3, mode="embedding")
tampilkan_hasil(hasil_em)

# Cek overlap
ids_tf = {h["case_id"] for h in hasil_tf}
ids_em = {h["case_id"] for h in hasil_em}
overlap = ids_tf & ids_em
print(f"\n📊 Overlap kasus di top-3: {len(overlap)}/3 — {overlap}")
print(f"   (Makin besar overlap = dua metode makin konsisten)")

## Cell 11 — Simpan Model & Vectors

In [ ]:
# Simpan semua artefak yang dibutuhkan Tahap 4 & 5
print("💾 Menyimpan artefak retrieval...\n")

# TF-IDF vectorizer
joblib.dump(tfidf,    MODEL_DIR / "tfidf_vectorizer.pkl")
joblib.dump(X_tfidf, MODEL_DIR / "tfidf_matrix.pkl")
print(f"  ✅ TF-IDF vectorizer  → {MODEL_DIR / 'tfidf_vectorizer.pkl'}")
print(f"  ✅ TF-IDF matrix      → {MODEL_DIR / 'tfidf_matrix.pkl'}")

# SVM model
joblib.dump(svm, MODEL_DIR / "svm_model.pkl")
joblib.dump(le,  MODEL_DIR / "label_encoder.pkl")
print(f"  ✅ SVM model          → {MODEL_DIR / 'svm_model.pkl'}")
print(f"  ✅ Label encoder      → {MODEL_DIR / 'label_encoder.pkl'}")

# Embeddings (numpy)
np.save(MODEL_DIR / "embeddings.npy", embeddings)
print(f"  ✅ ST embeddings      → {MODEL_DIR / 'embeddings.npy'}")

# DataFrame kasus (dipakai Tahap 4)
df_save = df.drop(columns=["teks_full", "teks_repr", "teks_processed"], errors="ignore")
df_save.to_csv(MODEL_DIR / "cases_index.csv", index=False, encoding="utf-8-sig")
print(f"  ✅ Cases index        → {MODEL_DIR / 'cases_index.csv'}")

print(f"\n✅ Semua artefak tersimpan di: {MODEL_DIR}")

## Cell 12 — Evaluasi Awal: Precision@K

> Karena ground-truth belum tersedia secara lengkap, evaluasi ini  
> menggunakan **label similarity** sebagai proxy:  
> kasus dianggap relevan jika labelnya sama dengan mayoritas top-k.

In [ ]:
def precision_at_k(hasil: list, label_query: str, k: int = 5) -> float:
    """
    Hitung Precision@K berdasarkan kesamaan label.
    label_query: label yang dianggap benar untuk query ini
    """
    if not hasil:
        return 0.0
    relevan = sum(1 for h in hasil[:k] if h["label"] == label_query)
    return relevan / min(k, len(hasil))


# Evaluasi semua query uji
print(f"{'='*55}")
print(f"  EVALUASI PRECISION@K (PROXY VIA LABEL)")
print(f"{'='*55}")
print(f"  {'Query':<8} {'Label Query':<12} {'P@3 TF-IDF':>10} {'P@3 Embed':>10}")
print(f"  {'─'*45}")

precision_tfidf_list = []
precision_embed_list = []

for q in QUERIES_UJI:
    h_tf = retrieve(q["query_text"], k=3, mode="tfidf")
    h_em = retrieve(q["query_text"], k=3, mode="embedding")

    # Tentukan label query dari majority vote top hasil
    if h_tf:
        label_q = max(set(h["label"] for h in h_tf), key=lambda l: sum(1 for h in h_tf if h["label"]==l))
    else:
        label_q = "?"

    p_tf = precision_at_k(h_tf, label_q, k=3)
    p_em = precision_at_k(h_em, label_q, k=3)

    precision_tfidf_list.append(p_tf)
    precision_embed_list.append(p_em)

    print(f"  {q['query_id']:<8} {label_q:<12} {p_tf:>10.2f} {p_em:>10.2f}")

avg_tf = sum(precision_tfidf_list) / len(precision_tfidf_list)
avg_em = sum(precision_embed_list) / len(precision_embed_list)

print(f"  {'─'*45}")
print(f"  {'Rata-rata':<20} {avg_tf:>10.2f} {avg_em:>10.2f}")
print(f"{'='*55}")
print(f"\n  Catatan: Evaluasi penuh (dengan ground-truth label nyata)")
print(f"  akan dilakukan di Tahap 5 (05_evaluation.ipynb)")

# Simpan metrik awal
metrics_awal = {
    "precision_at_3_tfidf"    : round(avg_tf, 4),
    "precision_at_3_embedding": round(avg_em, 4),
    "n_queries"               : len(QUERIES_UJI),
    "n_cases"                 : len(df),
    "mode_terbaik"            : "tfidf" if avg_tf >= avg_em else "embedding",
}
with open(DATA_EVAL / "metrics_awal.json", "w", encoding="utf-8") as f:
    json.dump(metrics_awal, f, ensure_ascii=False, indent=2)

print(f"\n  Mode terbaik (proxy): {metrics_awal['mode_terbaik'].upper()}")
print(f"\n💾 Metrik awal: {DATA_EVAL / 'metrics_awal.json'}")

---
## ✅ Tahap 3 Selesai!

**Output yang dihasilkan:**
```
/models/
├── tfidf_vectorizer.pkl   ← TF-IDF vectorizer
├── tfidf_matrix.pkl       ← Matrix TF-IDF semua dokumen
├── svm_model.pkl          ← Model SVM terlatih
├── label_encoder.pkl      ← Label encoder
├── embeddings.npy         ← SentenceTransformer embeddings
└── cases_index.csv        ← Index kasus (tanpa teks_full)
/data/eval/
├── queries.json           ← 5 query uji
├── retrieval_results.json ← Hasil retrieval per query
└── metrics_awal.json      ← Precision@K awal
```

**Fungsi utama yang tersedia untuk Tahap 4:**
```python
retrieve(query, k=5, mode="tfidf")     # atau mode="embedding"
```

**Langkah selanjutnya:** Buka `04_solution_reuse.ipynb`